# Module 19 - Agent loops

Use this notebook after `tests/test_agent.py` is passing. It starts with deterministic fake-backend checks for parsing, planning, scratchpad rendering, loop control, error recovery, and duplicate-action stops. The final section loads ProdLM for live ReAct runs against a real instruction model.

The deliverable is an agent failure-mode catalog: a few concrete transcripts, what went wrong, and what you would change in the prompt, tools, loop, or stop conditions.

## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import json
import subprocess
import sys

from IPython.display import Markdown, display

from g2c.agent import (
    Action,
    Agent,
    AgentStep,
    Observation,
    Plan,
    Scratchpad,
    extract_plan,
    parse_react_step,
    render_plan_block,
    render_planning_prompt,
    render_system_prompt,
)
from g2c.inference import Backend, BackendInfo, InferenceResult, load_prodlm_backend, prodlm_manifest_exists
from g2c.notebook_extras.sampling import printable
from g2c.tools import (
    Tool,
    ToolRegistry,
    make_calculator,
    make_read_file,
    make_run_python,
    run_with_tools,
)

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

Run the agent tests before proceeding. In the clean scaffold this cell should fail until you implement the Module 19 TODOs in `g2c/agent/`.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_agent.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 19 agent tests are not passing yet."

## Display helpers

In [ ]:
def short(text: Any, limit: int = 180) -> str:
    rendered = printable(str(text)).replace("\n", "\\n")
    if len(rendered) <= limit:
        return rendered
    return rendered[: limit - 3] + "..."


def markdown_table(rows: list[dict[str, Any]], columns: list[str]) -> str:
    def cell(value: Any) -> str:
        text = str(value).replace("|", "\\|").replace("\n", "<br>")
        return text

    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = ["| " + " | ".join(cell(row.get(col, "")) for col in columns) + " |" for row in rows]
    return "\n".join([header, sep, *body])


def fake_inference(completion: str, *, prompt: str = "prompt") -> InferenceResult:
    info = BackendInfo(name="fake", model_id="fake-agent")
    return InferenceResult(
        prompt=prompt,
        completion=completion,
        prompt_tokens=len(prompt.split()),
        completion_tokens=len(completion.split()),
        latency_ms=1.0,
        backend=info,
    )


def show_parsed_step(label: str, text: str) -> None:
    parsed = parse_react_step(text)
    rows = [
        {
            "field": "thought",
            "value": parsed.thought,
        },
        {
            "field": "action",
            "value": "" if parsed.action is None else parsed.action.tool,
        },
        {
            "field": "arguments",
            "value": "" if parsed.action is None else json.dumps(parsed.action.arguments),
        },
        {
            "field": "final_answer",
            "value": "" if parsed.final_answer is None else parsed.final_answer,
        },
        {
            "field": "parse_error",
            "value": "" if parsed.parse_error is None else parsed.parse_error,
        },
    ]
    display(Markdown(f"### {label}\n\n" + markdown_table(rows, ["field", "value"])))


def show_plan(plan: Plan | None) -> None:
    if plan is None:
        display(Markdown("No plan parsed."))
        return
    rows = [{"part": "goal", "text": plan.goal}]
    for index, step in enumerate(plan.steps, start=1):
        rows.append({"part": str(index), "text": step})
    display(Markdown(markdown_table(rows, ["part", "text"])))


def show_agent_run(result, *, backend: Backend | None = None, show_prompts: bool = False) -> None:
    print("stopped:", result.stopped_reason)
    print("final answer:", printable(result.final_answer or "(none)"))
    if result.plan is not None:
        print("plan goal:", result.plan.goal)
    print()

    rows = []
    for index, step in enumerate(result.steps, start=1):
        rows.append(
            {
                "step": index,
                "thought": short(step.thought, 120),
                "action": "" if step.action is None else step.action.tool,
                "arguments": "" if step.action is None else json.dumps(step.action.arguments),
                "observation": "" if step.observation is None else short(step.observation.output, 160),
                "error": "" if step.observation is None else ("yes" if step.observation.is_error else "no"),
                "final": "" if step.final_answer is None else short(step.final_answer, 120),
                "parse_error": "" if step.parse_error is None else short(step.parse_error, 120),
            }
        )
    display(Markdown(markdown_table(rows, ["step", "thought", "action", "arguments", "observation", "error", "final", "parse_error"])))

    if backend is not None and hasattr(backend, "calls"):
        call_rows = []
        for index, call in enumerate(backend.calls, start=1):
            call_rows.append(
                {
                    "call": index,
                    "prompt chars": len(call["prompt"]),
                    "max_new_tokens": call["max_new_tokens"],
                    "temperature": call["temperature"],
                }
            )
        display(Markdown(markdown_table(call_rows, ["call", "prompt chars", "max_new_tokens", "temperature"])))
        if show_prompts:
            for index, call in enumerate(backend.calls, start=1):
                print(f"\n--- prompt {index} ---")
                print(short(call["prompt"], limit=1600))

## Build a local tool registry

The fake-backend exercises use the same registry a live ProdLM run will use: calculator, read-file, and Python execution. The read-file tool points at a small local directory created by this notebook.

In [ ]:
scratch_dir = repo_root / "data" / "module19-agent"
scratch_dir.mkdir(parents=True, exist_ok=True)

(scratch_dir / "numbers.txt").write_text("3\n7\n10\n20\n", encoding="utf-8")
(scratch_dir / "sales.csv").write_text(
    "item,units,price\nnotebook,3,12\npen,10,2\nbag,1,35\n",
    encoding="utf-8",
)
(scratch_dir / "long.txt").write_text("alpha " * 250, encoding="utf-8")

registry = ToolRegistry([
    make_calculator(),
    make_read_file(root=scratch_dir),
    make_run_python(timeout=3.0),
])

print("registered:", registry.names())
print("scratch dir:", scratch_dir)

In [ ]:
print(render_system_prompt(registry.tools)[:1600])

## Exercise 1 - Parse ReAct steps

Start with the wire format. A step is either `Thought` + `Action` + `Action Input`, or `Thought` + `Final Answer`. The parser should tolerate small formatting wobble but still reject malformed action inputs.

In [ ]:
action_text = """Thought: I should use exact arithmetic.
Action: calculator
Action Input: {"expression": "23 * 17"}"""
final_text = """Thought: I now know the final answer.
Final Answer: 391"""
bad_text = """Thought: I should calculate.
Action: calculator
Action Input: {not json}"""
both_text = """Thought: I have enough information.
Action: calculator
Action Input: {"expression": "2 + 2"}
Final Answer: 4"""

show_parsed_step("Action step", action_text)
show_parsed_step("Final answer step", final_text)
show_parsed_step("Bad action input", bad_text)
show_parsed_step("Final answer wins", both_text)

## Exercise 2 - Extract and render a plan

The plan is a soft prior. It gets rendered into the prompt, but the model is not forced to follow it if observations reveal a better path.

In [ ]:
plan_text = """Goal: Compute the average in numbers.txt.
1. Read numbers.txt.
2. Add the numbers and divide by the count.
3. Report the mean with the supporting arithmetic."""
plan = extract_plan(plan_text, "Read numbers.txt and compute the average.")
show_plan(plan)

if plan is not None:
    print(render_plan_block(plan.goal, plan.steps))

print("\nPlanning prompt excerpt:\n")
print(render_planning_prompt("Read numbers.txt and compute the average.", registry.tools)[:1200])

## Exercise 3 - Render the scratchpad

The scratchpad is the agent's working memory. Each prior thought, action, input, and observation is rendered back into the next prompt.

In [ ]:
sp = Scratchpad()
step = AgentStep(
    completion=action_text,
    thought="I should use exact arithmetic.",
    action=Action(tool="calculator", arguments={"expression": "23 * 17"}),
    observation=Observation(output="391", is_error=False),
    final_answer=None,
    parse_error=None,
    inference=fake_inference(action_text),
)
sp.append(step)
print(sp.render())

In [ ]:
error_completion = """Thought: I used the wrong argument.
Action: calculator
Action Input: {"expr": "23 * 17"}"""
error_step = AgentStep(
    completion=error_completion,
    thought="I used the wrong argument.",
    action=Action(tool="calculator", arguments={"expr": "23 * 17"}),
    observation=Observation(output="unknown arguments: ['expr']", is_error=True),
    final_answer=None,
    parse_error=None,
    inference=fake_inference(error_completion),
)
sp.append(error_step)
print(sp.render())

In [ ]:
small_sp = Scratchpad(max_chars=180)
small_sp.append(step)
small_sp.append(error_step)
print(small_sp.render())

## Exercise 4 - Run the agent with a fake backend

Before involving a real model, use deterministic completions. This isolates the loop contract: prompt, parse, dispatch, observe, render scratchpad, repeat.

In [ ]:
class FakeBackend(Backend):
    def __init__(self, completions: list[str], *, model_id: str = "fake-agent") -> None:
        self._completions = list(completions)
        self._info = BackendInfo(name="fake", model_id=model_id)
        self.calls: list[dict[str, Any]] = []

    @property
    def info(self) -> BackendInfo:
        return self._info

    def complete(
        self,
        prompt: str,
        *,
        max_new_tokens: int = 128,
        temperature: float = 1.0,
        top_k: int | None = None,
        top_p: float | None = None,
    ) -> InferenceResult:
        if not self._completions:
            raise RuntimeError("FakeBackend has no completions left")
        completion = self._completions.pop(0)
        self.calls.append(
            {
                "prompt": prompt,
                "max_new_tokens": max_new_tokens,
                "temperature": temperature,
                "top_k": top_k,
                "top_p": top_p,
            }
        )
        return InferenceResult(
            prompt=prompt,
            completion=completion,
            prompt_tokens=len(prompt.split()),
            completion_tokens=len(completion.split()),
            latency_ms=1.0,
            backend=self._info,
        )

In [ ]:
fake_backend = FakeBackend(
    [
        """Thought: I should calculate exactly.
Action: calculator
Action Input: {"expression": "23 * 17"}""",
        """Thought: I now know the final answer.
Final Answer: 23 * 17 = 391.""",
    ]
)

fake_agent = Agent(fake_backend, registry, plan=False, max_steps=4, temperature=0.0)
fake_result = fake_agent.run("What is 23 times 17?")
show_agent_run(fake_result, backend=fake_backend)

print("Second prompt includes the first observation:")
print(short(fake_backend.calls[-1]["prompt"], limit=1200))

## Exercise 5 - Recover from a tool error

The first action uses the wrong argument name. The dispatcher turns validation failure into an error observation, and the next model turn can correct itself.

In [ ]:
recovery_backend = FakeBackend(
    [
        """Thought: I should calculate exactly.
Action: calculator
Action Input: {"expr": "23 * 17"}""",
        """Thought: The observation says the argument name was wrong.
Action: calculator
Action Input: {"expression": "23 * 17"}""",
        """Thought: I now know the final answer.
Final Answer: 391.""",
    ],
    model_id="fake-recovery",
)

recovery_agent = Agent(recovery_backend, registry, plan=False, max_steps=5, temperature=0.0)
recovery_result = recovery_agent.run("What is 23 times 17?")
show_agent_run(recovery_result, backend=recovery_backend)

## Exercise 6 - Stress-test loop detection

Duplicate-action detection is a heuristic: same tool, same arguments, two steps in a row. It catches a common runaway pattern, but you can disable it for legitimate retry workflows.

In [ ]:
duplicate_completion = """Thought: I should try the same calculation again.
Action: calculator
Action Input: {"expression": "2 + 2"}"""
looping_backend = FakeBackend([duplicate_completion, duplicate_completion, duplicate_completion], model_id="fake-loop")
looping_agent = Agent(looping_backend, registry, plan=False, max_steps=5, loop_detection=True)
looping_result = looping_agent.run("What is 2 + 2?")
show_agent_run(looping_result, backend=looping_backend)

In [ ]:
retry_backend = FakeBackend(
    [
        duplicate_completion,
        duplicate_completion,
        """Thought: I now know the final answer.
Final Answer: 4.""",
    ],
    model_id="fake-retry",
)
retry_agent = Agent(retry_backend, registry, plan=False, max_steps=5, loop_detection=False)
retry_result = retry_agent.run("What is 2 + 2?")
show_agent_run(retry_result, backend=retry_backend)

## Exercise 7 - Add a planning phase

With `plan=True`, the backend is called once before the main loop to produce a numbered plan. The plan is then visible in every ReAct prompt.

In [ ]:
planned_backend = FakeBackend(
    [
        """Goal: Compute the average from numbers.txt.
1. Read numbers.txt.
2. Use arithmetic to compute the average.
3. Report the result.""",
        """Thought: I need the file contents first.
Action: read_file
Action Input: {"path": "numbers.txt"}""",
        """Thought: I should compute the mean from the observed numbers.
Action: calculator
Action Input: {"expression": "(3 + 7 + 10 + 20) / 4"}""",
        """Thought: I now know the final answer.
Final Answer: The average is 10.""",
    ],
    model_id="fake-planned",
)

planned_agent = Agent(planned_backend, registry, plan=True, max_steps=5, temperature=0.0)
planned_result = planned_agent.run("Read numbers.txt and tell me the average.")
show_plan(planned_result.plan)
show_agent_run(planned_result, backend=planned_backend)

print("First ReAct prompt after the planning call:")
print(short(planned_backend.calls[1]["prompt"], limit=1600))

## Exercise 8 - Scratchpad cap

A character cap drops old scratchpad blocks when the rendered history grows too long. This is crude, but it makes the context-management issue concrete.

In [ ]:
cap_backend = FakeBackend(
    [
        """Thought: I should read the long file.
Action: read_file
Action Input: {"path": "long.txt", "max_chars": 1200}""",
        """Thought: I should do a tiny calculation.
Action: calculator
Action Input: {"expression": "10 + 5"}""",
        """Thought: I now know the final answer.
Final Answer: I read the long file and computed 15.""",
    ],
    model_id="fake-cap",
)

cap_agent = Agent(cap_backend, registry, plan=False, max_steps=4, scratchpad_max_chars=450)
cap_result = cap_agent.run("Read long.txt, then compute 10 + 5.")
show_agent_run(cap_result, backend=cap_backend)

for i, call in enumerate(cap_backend.calls, start=1):
    print(f"prompt {i}: {len(call['prompt'])} characters")

## Exercise 9 - Load ProdLM for live agent runs

The fake backend proves your contracts. ProdLM tests whether a real local instruction model follows the ReAct format. Run `./prodlm.sh` first if this cell says ProdLM is not configured.

In [ ]:
RUN_PRODLM_EXAMPLES = prodlm_manifest_exists(repo_root=repo_root)
PRODLM_MODEL_ID = None  # Optional override, e.g. "llama3.2:3b".

prodlm_backend = None
if RUN_PRODLM_EXAMPLES:
    prodlm_backend = load_prodlm_backend(repo_root=repo_root, model_id=PRODLM_MODEL_ID, required=False)
    print("loaded ProdLM:", prodlm_backend.info)
else:
    print("ProdLM is not configured. Run ./prodlm.sh, then rerun this cell.")

In [ ]:
def run_live_agent(
    question: str,
    *,
    tools: ToolRegistry = registry,
    plan: bool = True,
    max_steps: int = 6,
    max_new_tokens: int = 384,
    temperature: float = 0.0,
    scratchpad_max_chars: int | None = None,
):
    if prodlm_backend is None:
        print("Skipping live run: ProdLM is not configured.")
        return None
    agent = Agent(
        prodlm_backend,
        tools,
        plan=plan,
        max_steps=max_steps,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        scratchpad_max_chars=scratchpad_max_chars,
    )
    try:
        result = agent.run(question)
    except Exception as exc:  # Live local servers can be offline or mid-pull.
        print(type(exc).__name__ + ":", exc)
        return None
    show_agent_run(result)
    return result

## Exercise 10 - Live one-shot calculator

This should be an easy case. The task needs one tool call, so `plan=False` is usually cleaner.

In [ ]:
live_calc_question = "Use the calculator to compute (1847 * 29) - 138. Then give the final integer."
live_calc_result = run_live_agent(
    live_calc_question,
    tools=ToolRegistry([make_calculator()]),
    plan=False,
    max_steps=4,
)

## Exercise 11 - Live multi-step file task

Now give the model a task where it must read local data before answering. Try `plan=False` and `plan=True`; the difference is often visible in how direct the tool sequence is.

In [ ]:
file_question = "Read numbers.txt and report both the sum and the mean. Use tools for the file read and arithmetic."
file_result_without_plan = run_live_agent(file_question, plan=False, max_steps=6)
file_result_with_plan = run_live_agent(file_question, plan=True, max_steps=6)

## Exercise 12 - Compare Module 18 and Module 19

Module 18's loop waits for `<tool_call>` blocks. Module 19 asks the model for `Thought` / `Action` / `Action Input` turns. Run both on the same task and compare reliability, prompt length, and final answer quality.

In [ ]:
comparison_question = "Read sales.csv and compute total revenue."
comparison_registry = ToolRegistry([
    make_calculator(),
    make_read_file(root=scratch_dir),
    make_run_python(timeout=3.0),
])

if prodlm_backend is None:
    print("Skipping comparison: ProdLM is not configured.")
else:
    try:
        tool_loop_result = run_with_tools(
            prodlm_backend,
            comparison_registry,
            comparison_question,
            max_steps=5,
            max_new_tokens=384,
            temperature=0.0,
        )
        print("Module 18 stopped:", tool_loop_result.stopped_reason)
        print(printable(tool_loop_result.final_answer or "(none)"))
    except Exception as exc:
        print("Module 18 loop failed:", type(exc).__name__ + ":", exc)

    print("\nModule 19 agent:")
    agent_result = run_live_agent(comparison_question, tools=comparison_registry, plan=True, max_steps=6)

## Exercise 13 - Build a failure-mode catalog

Run this optional cell with your local model after the earlier live cells work. Keep a short catalog of what happened: parse failures, wrong tool choice, repeated actions, premature final answers, or good recoveries from tool errors.

In [ ]:
RUN_FAILURE_SESSION = False
failure_questions = [
    "Read numbers.txt and compute the largest number minus the smallest number.",
    "Read sales.csv and compute total revenue. Then explain the arithmetic.",
    "Use the calculator to compute 19 ** 3 + 7 ** 2.",
    "Try to read missing.txt. If it fails, explain what failed instead of inventing contents.",
]

failure_runs = []
if RUN_FAILURE_SESSION:
    for question in failure_questions:
        print("=" * 72)
        print(question)
        result = run_live_agent(question, tools=registry, plan=True, max_steps=6, temperature=0.0)
        failure_runs.append((question, result))
else:
    print("Set RUN_FAILURE_SESSION = True when you want to collect live examples.")

## Postmortem notes

Use this structure for your deliverable:

1. **Best success case:** paste the task, final answer, and the step table. Explain why it worked.
2. **Most informative failure:** paste the task and the step table. Categorize the failure: bad parse, wrong tool, invalid arguments, duplicate action, missing context, premature final answer, or model refusal.
3. **One loop change:** propose a change to the prompt, parser, tools, scratchpad cap, planning toggle, or stop condition.
4. **Module 18 comparison:** for one task, say whether the plain tool loop or ReAct agent was easier for the model to use.

The goal is not to prove the agent is robust. The goal is to see exactly where the wrapper helps and where the model is still the bottleneck.